# CIC3: Community Structure Sweep

Investigates how community structure (inter- and intra-community connectivity)
affects seeding strategy performance under the CIC3 metric.

**Design:** We use the SBM-Manual topology as a template (52 small communities
+ 1 hub community) and sweep two independent axes:

- **Intra-community connection** (`p_intra`): probability of edges within each
  of the 52 regular communities. Ranges from sparse (0.1) to fully connected
  (1.0).
- **Inter-community connection** (`p_inter`): probability of edges between
  pairs of regular communities. Ranges from near-isolated (0.001) to dense
  (~0.15), at which point community structure effectively vanishes.

**Fixed:** Hub community parameters (size=32, p_intra=0.5, p_inter_to_others=0.15)
are held constant so we isolate the effect of community structure/sparseness
rather than degree distribution.

**Output:**
1. One heatmap per strategy: `A_g^td` as a function of (p_inter, p_intra).
2. Line plots: fix p_intra at a high value and sweep p_inter (all strategies
   on one plot); fix p_inter at a low value and sweep p_intra.


In [1]:
import sys
sys.path.insert(0, '..')

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import itertools

from scm import SBMGenerator, CIC3Simulator
from scm import (
    MultiRandomSeeding, MultiHighDegreeSeeding, MultiHighSimplexSeeding,
    CommunityLouvainSeeding, CommunityFarthestFirstSeeding,
)
from scm.analysis import (
    attainment, time_discounted_attainment,
    exponential_decay, no_decay,
)

STRATEGIES = [
    ("Random", MultiRandomSeeding),
    ("High Degree", MultiHighDegreeSeeding),
    ("High 2-Simplex", MultiHighSimplexSeeding),
    ("Louvain", CommunityLouvainSeeding),
    ("Farthest-First", CommunityFarthestFirstSeeding),
]

## Shared parameters

In [ ]:
# --- Network ---
N = 500
TOPO_SEED = 2025

# Community structure: 52 small (size 9) + 1 hub (size 32)
REGULAR_K = 52
REGULAR_SIZE = 9
HUB_SIZE = 32
COMMUNITY_SIZES = [REGULAR_SIZE] * REGULAR_K + [HUB_SIZE]
assert sum(COMMUNITY_SIZES) == N
TOTAL_K = len(COMMUNITY_SIZES)  # 53

# Hub is fixed throughout the sweep
HUB_INTRA_P = 0.5
HUB_EDGE_P = 0.15
HUB_TRI_P = 0.04

# Triangle probability for regular communities (fixed)
REGULAR_TRI_P = 0.2

# --- Contagion ---
C = 50
BETA = 0.04
BETA_DELTA = 0.03
BETAS = [BETA] * C
BETA_DELTAS = [BETA_DELTA] * C

# --- Quotas (sum to N, no slack) ---
def make_quotas_and_seeds(N_topo, C=C, seed_frac_divisor=5):
    base = N_topo // C
    rem = N_topo - base * C
    quotas = [base + 1] * rem + [base] * (C - rem)
    seeds = [max(1, q // seed_frac_divisor) for q in quotas]
    assert sum(quotas) == N_topo
    return quotas, seeds

QUOTAS, SEEDS_PER_CONTAGION = make_quotas_and_seeds(N)

# --- Simulation ---
T_MAX = 200
V = no_decay()
NUM_TRIALS = 5  # fewer trials since we have many grid points

# --- Sweep grid ---
# Intra: from sparse communities to full cliques
P_INTRA_VALUES = np.array([0.1, 0.2, 0.4, 0.6, 0.8, 1.0])
# Inter: from near-isolated to very connected (community structure dissolves)
P_INTER_VALUES = np.array([0.001, 0.01, 0.05, 0.15, 0.3, 0.5])

print(f"Grid: {len(P_INTRA_VALUES)} intra x {len(P_INTER_VALUES)} inter = "
      f"{len(P_INTRA_VALUES) * len(P_INTER_VALUES)} topology configs")
print(f"Per config: {len(STRATEGIES)} strategies x {NUM_TRIALS} trials")
print(f"Total simulations: "
      f"{len(P_INTRA_VALUES) * len(P_INTER_VALUES) * len(STRATEGIES) * NUM_TRIALS}")

## Topology factory

Build an SBM with the given `p_intra` (regular communities) and `p_inter`
(between regular communities). Hub parameters are fixed.

In [ ]:
def make_block_matrix(p_intra, p_inter):
    """Build the (53x53) block matrix for given intra/inter probabilities."""
    K = TOTAL_K
    bm = np.full((K, K), p_inter)  # default: inter-regular connectivity
    # Regular community diagonals
    for a in range(REGULAR_K):
        bm[a, a] = p_intra
    # Hub community
    bm[-1, -1] = HUB_INTRA_P
    bm[-1, :-1] = HUB_EDGE_P
    bm[:-1, -1] = HUB_EDGE_P
    return bm.tolist()


def make_triangle_probs():
    """Triangle probabilities: fixed across the sweep."""
    return [REGULAR_TRI_P] * REGULAR_K + [HUB_TRI_P]


def generate_topology(p_intra, p_inter, seed=TOPO_SEED):
    """Generate an SBM topology for the given sweep parameters."""
    gen = SBMGenerator(
        community_sizes=COMMUNITY_SIZES,
        block_matrix=make_block_matrix(p_intra, p_inter),
        triangle_block_probs=make_triangle_probs(),
    )
    links, triangles = gen.generate(seed=seed)
    return {
        "links": links,
        "triangles": triangles,
        "N": N,
        "quotas": QUOTAS,
        "seeds": SEEDS_PER_CONTAGION,
        "k_avg": gen.k_avg,
        "k_d_avg": gen.k_delta_avg,
    }

## Run the sweep

For each (p_intra, p_inter) grid point, generate the topology once and run
all strategies for NUM_TRIALS each. Results are cached to disk so the
notebook can be re-run without re-computing.

In [ ]:
RESULTS_DIR = "results/community_sweep"
os.makedirs(RESULTS_DIR, exist_ok=True)


def run_trial(topo, strat_cls):
    """Run a single CIC3 trial and return metrics."""
    seeder = strat_cls(
        N=topo["N"], num_seeds_per_contagion=topo["seeds"],
        links=topo["links"], triangles=topo["triangles"],
    )
    seeds = seeder.seed()
    sim = CIC3Simulator(
        links=topo["links"], triangles=topo["triangles"],
        initial_infected_per_contagion=seeds,
        betas=BETAS, beta_deltas=BETA_DELTAS, quotas=topo["quotas"],
    )
    sim.run(T_MAX)
    A_i, A_g = attainment(sim.infected_by, topo["quotas"])
    A_i_td, A_g_td = time_discounted_attainment(
        sim.infected_by, sim.infection_times, topo["quotas"], V,
    )
    return {"A_g": A_g, "A_g_td": A_g_td, "A_i": A_i, "A_i_td": A_i_td}


def cache_key(p_intra, p_inter, strat_name):
    return f"intra{p_intra:.3f}_inter{p_inter:.4f}_{strat_name}"


# Master results dict: results[p_intra][p_inter][strat_name] = list of trial dicts
all_results = {}

total_configs = len(P_INTRA_VALUES) * len(P_INTER_VALUES)
config_idx = 0

for p_intra in P_INTRA_VALUES:
    all_results[p_intra] = {}
    for p_inter in P_INTER_VALUES:
        config_idx += 1
        print(f"\n[{config_idx}/{total_configs}] "
              f"p_intra={p_intra:.2f}, p_inter={p_inter:.4f}")

        # Generate topology for this grid point
        topo = generate_topology(p_intra, p_inter)
        print(f"  Realized k_avg={topo['k_avg']:.2f}, "
              f"k_delta_avg={topo['k_d_avg']:.2f}")

        strat_results = {}
        for strat_name, strat_cls in STRATEGIES:
            key = cache_key(p_intra, p_inter, strat_name)
            cache_path = os.path.join(RESULTS_DIR, f"{key}.pkl")

            if os.path.exists(cache_path):
                with open(cache_path, "rb") as f:
                    trials = pickle.load(f)
                print(f"  [{strat_name}] loaded from cache")
            else:
                trials = []
                for t in range(NUM_TRIALS):
                    trials.append(run_trial(topo, strat_cls))
                with open(cache_path, "wb") as f:
                    pickle.dump(trials, f)
                print(f"  [{strat_name}] {NUM_TRIALS} trials done")

            strat_results[strat_name] = trials
        all_results[p_intra][p_inter] = strat_results

print("\n=== Sweep complete ===")

## Aggregate results

For each (p_intra, p_inter, strategy) cell, compute mean `A_g^td` across trials.

In [ ]:
strat_names = [s[0] for s in STRATEGIES]

# Build 2D arrays for heatmaps: shape (len(P_INTER_VALUES), len(P_INTRA_VALUES))
# Convention: rows = inter (y-axis), cols = intra (x-axis)
heatmaps = {}  # strat_name -> 2D numpy array of mean A_g_td

for strat_name in strat_names:
    grid = np.zeros((len(P_INTER_VALUES), len(P_INTRA_VALUES)))
    for ii, p_intra in enumerate(P_INTRA_VALUES):
        for ji, p_inter in enumerate(P_INTER_VALUES):
            trials = all_results[p_intra][p_inter][strat_name]
            mean_td = np.mean([tr["A_g_td"] for tr in trials])
            grid[ji, ii] = mean_td
    heatmaps[strat_name] = grid

print("Aggregation done. Heatmap shape:", grid.shape)

## Plot: Heatmaps (one per strategy)

Each heatmap shows mean `A_g^td` with intra-community connectivity on the
x-axis and inter-community connectivity on the y-axis. A shared colorbar
makes cross-strategy comparison straightforward.

In [ ]:
# Find global min/max for consistent color scale
vmin = min(h.min() for h in heatmaps.values())
vmax = max(h.max() for h in heatmaps.values())

ncols = 3
nrows = int(np.ceil(len(strat_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.flatten()

for i, strat_name in enumerate(strat_names):
    ax = axes_flat[i]
    im = ax.imshow(
        heatmaps[strat_name], aspect="auto", origin="lower",
        vmin=vmin, vmax=vmax, cmap="viridis",
        extent=[
            P_INTRA_VALUES[0], P_INTRA_VALUES[-1],
            P_INTER_VALUES[0], P_INTER_VALUES[-1],
        ],
    )
    ax.set_title(strat_name, fontsize=11)
    ax.set_xlabel(r"Intra-community $p_{intra}$")
    ax.set_ylabel(r"Inter-community $p_{inter}$")

# Turn off extra axes
for j in range(len(strat_names), len(axes_flat)):
    axes_flat[j].axis("off")

# Shared colorbar
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label=r"Mean $A_g^{td}$")

fig.suptitle(
    f"CIC3 strategy performance vs community structure\n"
    f"(N={N}, C={C}, trials={NUM_TRIALS})",
    fontsize=13, y=1.02,
)
fig.tight_layout(rect=[0, 0, 0.88, 1.0])
fig.savefig(
    "../figures/cic3_community_sweep_heatmaps.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Plot: Line sweep — fix p_intra, vary p_inter

Fix intra-community connection at a high value (strong communities) and
sweep inter-community connection. All strategies plotted together.

In [ ]:
# Pick p_intra = 1.0 (full cliques) as the fixed point
FIXED_INTRA = P_INTRA_VALUES[-1]  # 1.0
intra_idx = len(P_INTRA_VALUES) - 1

strat_colors = plt.cm.tab10(np.linspace(0, 1, 10))[:len(strat_names)]

fig, ax = plt.subplots(figsize=(8, 5))
for si, strat_name in enumerate(strat_names):
    means = []
    stds = []
    for ji, p_inter in enumerate(P_INTER_VALUES):
        trials = all_results[FIXED_INTRA][p_inter][strat_name]
        vals = [tr["A_g_td"] for tr in trials]
        means.append(np.mean(vals))
        stds.append(np.std(vals, ddof=1) if len(vals) > 1 else 0.0)
    means = np.array(means)
    stds = np.array(stds)
    ax.plot(P_INTER_VALUES, means, color=strat_colors[si], lw=2,
            marker="o", label=strat_name)
    ax.fill_between(P_INTER_VALUES, means - stds, means + stds,
                    color=strat_colors[si], alpha=0.15)

ax.set_xlabel(r"Inter-community connection $p_{inter}$", fontsize=11)
ax.set_ylabel(r"Mean $A_g^{td}$", fontsize=11)
ax.set_title(
    f"Strategy performance vs inter-community connectivity\n"
    rf"(fixed $p_{{intra}}={FIXED_INTRA}$, N={N}, C={C}, trials={NUM_TRIALS})",
    fontsize=12,
)
ax.legend(loc="best")
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(
    "../figures/cic3_sweep_inter_fixed_intra.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Plot: Line sweep — fix p_inter, vary p_intra

Fix inter-community connection at a low value (well-separated communities)
and sweep intra-community density.

In [ ]:
# Pick p_inter = 0.001 (near-isolated communities) as the fixed point
FIXED_INTER = P_INTER_VALUES[0]  # 0.001

fig, ax = plt.subplots(figsize=(8, 5))
for si, strat_name in enumerate(strat_names):
    means = []
    stds = []
    for ii, p_intra in enumerate(P_INTRA_VALUES):
        trials = all_results[p_intra][FIXED_INTER][strat_name]
        vals = [tr["A_g_td"] for tr in trials]
        means.append(np.mean(vals))
        stds.append(np.std(vals, ddof=1) if len(vals) > 1 else 0.0)
    means = np.array(means)
    stds = np.array(stds)
    ax.plot(P_INTRA_VALUES, means, color=strat_colors[si], lw=2,
            marker="o", label=strat_name)
    ax.fill_between(P_INTRA_VALUES, means - stds, means + stds,
                    color=strat_colors[si], alpha=0.15)

ax.set_xlabel(r"Intra-community connection $p_{intra}$", fontsize=11)
ax.set_ylabel(r"Mean $A_g^{td}$", fontsize=11)
ax.set_title(
    f"Strategy performance vs intra-community connectivity\n"
    rf"(fixed $p_{{inter}}={FIXED_INTER}$, N={N}, C={C}, trials={NUM_TRIALS})",
    fontsize=12,
)
ax.legend(loc="best")
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(
    "../figures/cic3_sweep_intra_fixed_inter.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Plot: Difference heatmaps (strategy vs Random)

Shows where each non-random strategy beats or loses to the Random baseline.
Positive (blue) = strategy is better; negative (red) = Random is better.

In [ ]:
# Compute difference grids
diff_heatmaps = {}
for strat_name in strat_names:
    if strat_name == "Random":
        continue
    diff_heatmaps[strat_name] = heatmaps[strat_name] - heatmaps["Random"]

# Symmetric color range
diff_max = max(abs(d).max() for d in diff_heatmaps.values())

ncols = 2
nrows = int(np.ceil(len(diff_heatmaps) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.flatten()

for i, (strat_name, diff) in enumerate(diff_heatmaps.items()):
    ax = axes_flat[i]
    im = ax.imshow(
        diff, aspect="auto", origin="lower",
        vmin=-diff_max, vmax=diff_max, cmap="RdBu",
        extent=[
            P_INTRA_VALUES[0], P_INTRA_VALUES[-1],
            P_INTER_VALUES[0], P_INTER_VALUES[-1],
        ],
    )
    ax.set_title(f"{strat_name} − Random", fontsize=11)
    ax.set_xlabel(r"Intra-community $p_{intra}$")
    ax.set_ylabel(r"Inter-community $p_{inter}$")

for j in range(len(diff_heatmaps), len(axes_flat)):
    axes_flat[j].axis("off")

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label=r"$\Delta A_g^{td}$ (strategy − Random)")

fig.suptitle(
    f"Strategy advantage over Random baseline\n"
    f"(blue = strategy better, red = Random better)",
    fontsize=13, y=1.02,
)
fig.tight_layout(rect=[0, 0, 0.88, 1.0])
fig.savefig(
    "../figures/cic3_community_sweep_diff_heatmaps.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Summary table

Print the best strategy for each grid point.

In [ ]:
print(f"{'p_intra':<10s}{'p_inter':<10s}{'Best Strategy':<20s}{'A_g_td':<10s}")
print("-" * 50)
for ii, p_intra in enumerate(P_INTRA_VALUES):
    for ji, p_inter in enumerate(P_INTER_VALUES):
        best_strat = None
        best_val = -1
        for strat_name in strat_names:
            val = heatmaps[strat_name][ji, ii]
            if val > best_val:
                best_val = val
                best_strat = strat_name
        print(f"{p_intra:<10.3f}{p_inter:<10.4f}{best_strat:<20s}{best_val:<10.4f}")